In [4]:
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [5]:
PROJECT_ROOT = Path.cwd().parent
gangnam_df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "gangnam" / "gangnam_2016_2025.csv")
seocho_df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "seocho" / "seocho_2016_2025.csv")

df = pd.concat([gangnam_df, seocho_df], ignore_index=True)
print(df.shape)
df.info()

(69686, 34)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69686 entries, 0 to 69685
Data columns (total 34 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   aptDong           15520 non-null  object 
 1   aptNm             69686 non-null  object 
 2   aptSeq            69686 non-null  object 
 3   bonbun            69686 non-null  int64  
 4   bubun             69686 non-null  int64  
 5   buildYear         69686 non-null  int64  
 6   buyerGbn          14324 non-null  object 
 7   cdealDay          1943 non-null   object 
 8   cdealType         1943 non-null   object 
 9   dealAmount        69686 non-null  object 
 10  dealDay           69686 non-null  int64  
 11  dealMonth         69686 non-null  int64  
 12  dealYear          69686 non-null  int64  
 13  dealingGbn        20164 non-null  object 
 14  estateAgentSggNm  19295 non-null  object 
 15  excluUseAr        69686 non-null  float64
 16  floor             69686 non-

In [3]:
df["dealAmount"] = (
    df["dealAmount"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
    .astype(int)
)
df["dealAmount"].describe()

count    6.968600e+04
mean     1.764793e+05
std      1.164628e+05
min      1.100000e+04
25%      9.662500e+04
50%      1.500000e+05
75%      2.260000e+05
max      1.900000e+06
Name: dealAmount, dtype: float64

In [9]:
df["deal_year"] = df["deal_ymd"].astype(str).str[:4].astype(int)

for col in ["dealingGbn", "buyerGbn", "slerGbn"]:
    print(col)
    print(df.groupby("deal_year")[col].apply(lambda x: x.notna().mean()))
    print()

dealingGbn
deal_year
2016    0.000000
2017    0.000000
2018    0.000000
2019    0.000000
2020    0.000000
2021    0.077785
2022    1.000000
2023    1.000000
2024    1.000000
2025    1.000000
Name: dealingGbn, dtype: float64

buyerGbn
deal_year
2016    0.0
2017    0.0
2018    0.0
2019    0.0
2020    0.0
2021    0.0
2022    0.0
2023    0.0
2024    1.0
2025    1.0
Name: buyerGbn, dtype: float64

slerGbn
deal_year
2016    0.0
2017    0.0
2018    0.0
2019    0.0
2020    0.0
2021    0.0
2022    0.0
2023    0.0
2024    1.0
2025    1.0
Name: slerGbn, dtype: float64



In [6]:
def blank_or_na_ratio(series):
    is_blank = series.astype(str).str.strip().isin(["", "nan"])
    return is_blank.mean()

quality_check = pd.DataFrame({
    "결측/공백 비율": df.apply(blank_or_na_ratio)
}).sort_values("결측/공백 비율", ascending=False)

quality_check

,결측/공백 비율
cdealDay,0.972118
cdealType,0.972118
slerGbn,0.794449
buyerGbn,0.794449
aptDong,0.777287
rgstDate,0.755848
estateAgentSggNm,0.723115
dealingGbn,0.710645
roadNmbCd,0.058290
roadNmBonbun,0.000000


In [7]:
df["is_cancelled"] = df["cdealType"].astype(str).str.strip() == "O"
print("해제거래 건수:", df["is_cancelled"].sum())
print("해제거래 비율:", df["is_cancelled"].mean() * 100, "%")

df[df["is_cancelled"]][["aptNm", "dealYear", "dealMonth", "dealAmount", "cdealDay"]].head()

해제거래 건수: 1943
해제거래 비율: 2.788221450506558 %


,aptNm,dealYear,dealMonth,dealAmount,cdealDay
22070,청담현대3차아파트,2020,2,"237,000",20.03.23
22082,도곡지웰카운티101동,2020,2,"166,000",20.04.06
22110,신동아,2020,2,"101,000",20.03.24
22117,개포주공1단지,2020,2,"218,000",20.03.02
22127,성원대치2단지아파트,2020,2,"123,500",20.03.25


In [8]:
df["dealYear"] = df["dealYear"].astype(int)
pivot = df.pivot_table(
    index="dealYear", columns="gu_name", values="aptNm", aggfunc="count"
)
pivot

gu_name,gangnam,seocho
dealYear,,
2016,6706,4842
2017,7038,5263
2018,3529,3106
2019,4628,3233
2020,3717,3346
2021,2259,2202
2022,879,690
2023,2339,1585
2024,3754,3105
